# 04 — Failure Case Analysis

Load the KITTI-C results CSV and produce:
- Per-corruption summary tables
- Severity curves
- Worst / median / best image galleries

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2

from src.analysis.failure_slices import get_worst_n, get_best_n, get_median_n
from src.analysis.report_tables import corruption_summary_table, severity_curve, per_corruption_severity_pivot

RESULTS_CSV = PROJECT_ROOT / 'outputs' / 'metrics' / 'kittic_results.csv'
GALLERY_DIR = PROJECT_ROOT / 'outputs' / 'galleries' / 'kitti_c'

df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df)} results')
df.head()

In [ ]:
print('=== Corruption summary (mean abs_rel) ===')
corruption_summary_table(df)[['abs_rel', 'rmse', 'delta1']]

In [ ]:
print('=== Per-corruption severity pivot (abs_rel) ===')
per_corruption_severity_pivot(df, metric='abs_rel')

In [ ]:
# Severity curves for all corruption types
corruptions = sorted(df['corruption_type'].unique())

fig, axes = plt.subplots(1, min(len(corruptions), 4), figsize=(16, 4), sharey=True)
if len(corruptions) == 1:
    axes = [axes]

for ax, ct in zip(axes, corruptions[:4]):
    curve = severity_curve(df, ct)
    ax.plot(curve['severity'], curve['abs_rel'], marker='o')
    ax.set_title(ct)
    ax.set_xlabel('Severity')
    ax.set_ylabel('Abs Rel')
    ax.grid(True)

plt.suptitle('MiDaS Abs Rel vs. Corruption Severity', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gallery: worst 4 images for the highest-error corruption type
worst_corruption = corruption_summary_table(df)['abs_rel'].idxmax()
print(f'Worst corruption type: {worst_corruption}')

worst_rows = get_worst_n(df, metric='abs_rel', corruption_type=worst_corruption, n=4)

fig, axes = plt.subplots(len(worst_rows), 2, figsize=(14, 4 * len(worst_rows)))
if len(worst_rows) == 1:
    axes = [axes]

for i, (_, row) in enumerate(worst_rows.iterrows()):
    try:
        img = cv2.cvtColor(cv2.imread(row['image_path']), cv2.COLOR_BGR2RGB)
        depth = np.load(row['pred_path'])

        axes[i][0].imshow(img)
        axes[i][0].set_title(f"[{row['corruption_type']} sev={row['severity']}]\n{Path(row['image_path']).name}")
        axes[i][0].axis('off')

        axes[i][1].imshow(depth, cmap='inferno')
        axes[i][1].set_title(f"abs_rel={row['abs_rel']:.4f}  rmse={row['rmse']:.3f}")
        axes[i][1].axis('off')
    except Exception as e:
        print(f'Could not show row {i}: {e}')

plt.suptitle(f'Top Failure Cases — {worst_corruption}', fontsize=14)
plt.tight_layout()
plt.show()